# UniMate — A Simple University Assistant

A beginner-friendly Python notebook that demonstrates four core AI-assistant building blocks, **fully offline (no API calls, no API key):**

1. Tool calling — Pydantic schemas, a tool registry, and parallel tool execution
2. Document ingestion and chunking
3. RAG retrieval and evaluation
4. Conversation memory and context management

It also includes a small **ipywidgets dashboard** at the end.

### How to use
- Run the cells from top to bottom.
- Nothing in this notebook contacts an external service, so there are no rate limits or retry errors.


## 0. Install and import dependencies

The notebook uses a small set of common libraries. If your environment already has them, you can skip the install cell.

In [1]:
%pip -q install pydantic ipywidgets scikit-learn

In [2]:
import re
import json
from typing import Callable
from concurrent.futures import ThreadPoolExecutor, as_completed

from pydantic import BaseModel, Field, ValidationError

print("Setup complete.")

Setup complete.


## 1. Tools: from schema to a small plugin system

A tool is an action the assistant can run, such as searching course notes or calculating a result. The application validates the arguments and runs the matching function.

### 1.1 Define tool schemas with Pydantic

In [3]:
class SearchNotesArgs(BaseModel):
    query: str = Field(description="The topic or phrase to search for in course notes")

class CalculatorArgs(BaseModel):
    expression: str = Field(description="A basic arithmetic expression, such as 12 * (4 + 3)")

def pydantic_schema(model_class):
    # Works with Pydantic v2 and v1.
    if hasattr(model_class, "model_json_schema"):
        return model_class.model_json_schema()
    return model_class.schema()


print(json.dumps(pydantic_schema(SearchNotesArgs), indent=2))

{
  "properties": {
    "query": {
      "description": "The topic or phrase to search for in course notes",
      "title": "Query",
      "type": "string"
    }
  },
  "required": [
    "query"
  ],
  "title": "SearchNotesArgs",
  "type": "object"
}


### 1.2 Prepare a tiny university knowledge base

This same content will be used later by the ingestion and RAG sections.

In [4]:
course_documents = [
    {
        "source": "ML_Week1",
        "text": "Machine learning is a field of AI that enables systems to learn patterns from data. "
                "Supervised learning uses labeled examples. Classification predicts categories, "
                "while regression predicts numerical values."
    },
    {
        "source": "NLP_Week2",
        "text": "Natural language processing (NLP) focuses on interactions between computers and human language. "
                "Tokenization splits text into smaller units. Embeddings represent text as numerical vectors."
    },
    {
        "source": "RAG_Week3",
        "text": "Retrieval-augmented generation (RAG) retrieves relevant external information and provides it "
                "to a language model as context. A typical RAG pipeline includes ingestion, chunking, "
                "retrieval, prompt construction, and generation."
    },
    {
        "source": "API_Week4",
        "text": "An API client sends requests to a model endpoint. Retry logic can handle temporary errors. "
                "Streaming returns partial output as it is generated. TTFT means time to first token."
    },
]

def search_notes(query: str, top_k: int = 3):
    terms = set(re.findall(r"\w+", query.lower()))
    scored = []
    for doc in course_documents:
        words = set(re.findall(r"\w+", doc["text"].lower()))
        score = len(terms & words)
        if score:
            scored.append((score, doc))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [
        {"source": doc["source"], "text": doc["text"], "score": score}
        for score, doc in scored[:top_k]
    ]


def calculator(expression: str):
    # Deliberately limited arithmetic parser: no names, imports, or arbitrary Python.
    if not re.fullmatch(r"[0-9+\-*/(). %]+", expression):
        raise ValueError("Only basic arithmetic characters are allowed.")
    return eval(expression, {"__builtins__": {}}, {})

### 1.3 ToolRegistry

The registry connects a tool name, its validated argument schema, and the Python function that performs the action. This is a simple version of a plugin system.

In [5]:
class ToolRegistry:
    def __init__(self):
        self.tools = {}

    def register(self, name: str, description: str, args_model, function: Callable):
        self.tools[name] = {
            "description": description,
            "args_model": args_model,
            "function": function,
        }

    def tool_schemas(self):
        result = []
        for name, item in self.tools.items():
            result.append({
                "type": "function",
                "function": {
                    "name": name,
                    "description": item["description"],
                    "parameters": pydantic_schema(item["args_model"]),
                }
            })
        return result

    def execute(self, name: str, raw_arguments: str):
        if name not in self.tools:
            raise KeyError(f"Unknown tool: {name}")

        item = self.tools[name]
        data = json.loads(raw_arguments)

        # Validate arguments before calling the function.
        if hasattr(item["args_model"], "model_validate"):
            validated = item["args_model"].model_validate(data)
            args = validated.model_dump()
        else:
            validated = item["args_model"](**data)
            args = validated.dict()

        return item["function"](**args)


registry = ToolRegistry()
registry.register(
    "search_course_notes",
    "Search the university course notes for relevant information.",
    SearchNotesArgs,
    lambda query: search_notes(query),
)
registry.register(
    "calculate",
    "Calculate a basic arithmetic expression.",
    CalculatorArgs,
    calculator,
)

print("Registered tools:", list(registry.tools.keys()))

Registered tools: ['search_course_notes', 'calculate']


### 1.4 Run a tool

The registry validates the arguments before running the function, so bad input is caught early.

In [6]:
print(registry.execute("search_course_notes", json.dumps({"query": "What does TTFT mean?"})))
print(registry.execute("calculate", json.dumps({"expression": "12 * (4 + 3)"})))

# Invalid arguments are rejected by the Pydantic schema.
try:
    registry.execute("calculate", json.dumps({"wrong_field": "1+1"}))
except ValidationError as exc:
    print("Validation error caught:", exc.error_count(), "problem(s)")

[{'source': 'API_Week4', 'text': 'An API client sends requests to a model endpoint. Retry logic can handle temporary errors. Streaming returns partial output as it is generated. TTFT means time to first token.', 'score': 1}]
84
Validation error caught: 1 problem(s)


### 1.5 Parallel tool calls

Several independent tool calls can be executed concurrently with `ThreadPoolExecutor`.

This helper is useful when tool calls do not depend on each other's results.

In [7]:
def execute_tool_calls_in_parallel(tool_calls: list, max_workers: int = 4):
    def run_one(call):
        name = call["function"]["name"]
        arguments = call["function"]["arguments"]
        try:
            result = registry.execute(name, arguments)
        except Exception as exc:
            result = {"error": str(exc)}
        return {
            "tool_call_id": call["id"],
            "name": name,
            "result": result,
        }

    results = []
    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        futures = [pool.submit(run_one, call) for call in tool_calls]
        for future in as_completed(futures):
            results.append(future.result())
    return results


# Example: two independent tool calls running at the same time.
example_calls = [
    {
        "id": "call_search",
        "function": {
            "name": "search_course_notes",
            "arguments": json.dumps({"query": "RAG retrieval"})
        }
    },
    {
        "id": "call_calculate",
        "function": {
            "name": "calculate",
            "arguments": json.dumps({"expression": "15 * 4"})
        }
    },
]

parallel_results = execute_tool_calls_in_parallel(example_calls)
print(json.dumps(parallel_results, indent=2, ensure_ascii=False, default=str))

[
  {
    "tool_call_id": "call_search",
    "name": "search_course_notes",
    "result": [
      {
        "source": "RAG_Week3",
        "text": "Retrieval-augmented generation (RAG) retrieves relevant external information and provides it to a language model as context. A typical RAG pipeline includes ingestion, chunking, retrieval, prompt construction, and generation.",
        "score": 2
      }
    ]
  },
  {
    "tool_call_id": "call_calculate",
    "name": "calculate",
    "result": 60
  }
]


## 2. Ingestion and chunking pipeline

A RAG system needs to turn source documents into smaller pieces called **chunks**. Smaller chunks can make retrieval more focused, while overlap helps preserve context between neighboring chunks.

This simple example:
1. accepts documents,
2. normalizes whitespace,
3. splits text into chunks,
4. stores the source and chunk ID with every chunk.

In [8]:
def clean_text(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()


def chunk_text(text: str, chunk_size: int = 35, overlap: int = 7):
    if chunk_size <= 0:
        raise ValueError("chunk_size must be positive")
    if overlap < 0 or overlap >= chunk_size:
        raise ValueError("overlap must be between 0 and chunk_size - 1")

    words = clean_text(text).split()
    chunks = []
    step = chunk_size - overlap

    for start in range(0, len(words), step):
        chunk_words = words[start:start + chunk_size]
        if not chunk_words:
            break
        chunks.append(" ".join(chunk_words))
        if start + chunk_size >= len(words):
            break

    return chunks


def ingest_documents(documents, chunk_size=35, overlap=7):
    chunk_records = []
    for doc in documents:
        chunks = chunk_text(doc["text"], chunk_size, overlap)
        for index, chunk in enumerate(chunks):
            chunk_records.append({
                "chunk_id": f'{doc["source"]}_chunk_{index}',
                "source": doc["source"],
                "text": chunk,
            })
    return chunk_records


chunks = ingest_documents(course_documents, chunk_size=30, overlap=6)
print("Number of chunks:", len(chunks))
for item in chunks[:3]:
    print("\n", item)

Number of chunks: 4

 {'chunk_id': 'ML_Week1_chunk_0', 'source': 'ML_Week1', 'text': 'Machine learning is a field of AI that enables systems to learn patterns from data. Supervised learning uses labeled examples. Classification predicts categories, while regression predicts numerical values.'}

 {'chunk_id': 'NLP_Week2_chunk_0', 'source': 'NLP_Week2', 'text': 'Natural language processing (NLP) focuses on interactions between computers and human language. Tokenization splits text into smaller units. Embeddings represent text as numerical vectors.'}

 {'chunk_id': 'RAG_Week3_chunk_0', 'source': 'RAG_Week3', 'text': 'Retrieval-augmented generation (RAG) retrieves relevant external information and provides it to a language model as context. A typical RAG pipeline includes ingestion, chunking, retrieval, prompt construction, and generation.'}


## 3. RAG service and retrieval evaluation

For a lightweight notebook, retrieval uses **TF-IDF** rather than a hosted vector database. TF-IDF is easy to inspect. The same service structure could later be connected to embeddings and a vector database.

### 3.1 Build the retrieval index

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

class RAGService:
    def __init__(self, chunk_records):
        self.chunks = chunk_records
        self.vectorizer = TfidfVectorizer(stop_words="english")
        self.matrix = self.vectorizer.fit_transform(
            [item["text"] for item in self.chunks]
        )

    def retrieve(self, query: str, top_k: int = 3):
        query_vector = self.vectorizer.transform([query])
        scores = cosine_similarity(query_vector, self.matrix).flatten()
        indices = scores.argsort()[::-1][:top_k]

        results = []
        for idx in indices:
            if scores[idx] <= 0:
                continue
            item = dict(self.chunks[idx])
            item["score"] = float(scores[idx])
            results.append(item)
        return results

    def build_context(self, query: str, top_k: int = 3):
        results = self.retrieve(query, top_k)
        context = "\n\n".join(
            f'Source: {item["source"]}\n{item["text"]}'
            for item in results
        )
        return context, results

    def answer(self, query: str, top_k: int = 3):
        context, results = self.build_context(query, top_k)
        if not results:
            return "I couldn't find relevant information in the course notes.", results
        return "Relevant course notes:\n" + context, results


rag = RAGService(chunks)
answer, retrieved = rag.answer("What is RAG and what are its main stages?")
print(answer)
print("\nRetrieved sources:", [item["source"] for item in retrieved])

Relevant course notes:
Source: RAG_Week3
Retrieval-augmented generation (RAG) retrieves relevant external information and provides it to a language model as context. A typical RAG pipeline includes ingestion, chunking, retrieval, prompt construction, and generation.

Retrieved sources: ['RAG_Week3']


### 3.2 Retrieval evaluation: Recall@K and MRR

Evaluation checks whether retrieval returns the chunks that contain the expected information.

- **Recall@K:** Did at least one relevant chunk appear in the top K results? Across multiple questions, it is the proportion of questions for which that happened.
- **MRR (Mean Reciprocal Rank):** Rewards relevant results appearing near the top. If the first relevant result is ranked 1, its reciprocal rank is 1; if ranked 2, it is 0.5.

In [10]:
evaluation_set = [
    {
        "query": "What is supervised learning?",
        "relevant_sources": {"ML_Week1"},
    },
    {
        "query": "What are embeddings and tokenization?",
        "relevant_sources": {"NLP_Week2"},
    },
    {
        "query": "Explain retrieval augmented generation.",
        "relevant_sources": {"RAG_Week3"},
    },
    {
        "query": "What does TTFT measure?",
        "relevant_sources": {"API_Week4"},
    },
]


def evaluate_retrieval(rag_service, test_cases, k=3):
    hits = []
    reciprocal_ranks = []

    for case in test_cases:
        results = rag_service.retrieve(case["query"], top_k=k)
        relevant = case["relevant_sources"]

        ranks = [
            rank for rank, item in enumerate(results, start=1)
            if item["source"] in relevant
        ]

        hits.append(1 if ranks else 0)
        reciprocal_ranks.append(1 / ranks[0] if ranks else 0)

    return {
        f"Recall@{k}": sum(hits) / len(hits) if hits else 0,
        "MRR": sum(reciprocal_ranks) / len(reciprocal_ranks) if reciprocal_ranks else 0,
        "questions": len(test_cases),
    }


evaluation_results = evaluate_retrieval(rag, evaluation_set, k=3)
print(json.dumps(evaluation_results, indent=2))

{
  "Recall@3": 1.0,
  "MRR": 1.0,
  "questions": 4
}


## 4. Conversation memory and context management

A conversation can become too long to send in every model request. A simple memory manager keeps a system message and only the most recent messages.

This example uses a **sliding window**. It is intentionally simple; production systems may also summarize older messages or store long-term user preferences with consent.

In [11]:
class ConversationMemory:
    def __init__(self, system_prompt: str, max_messages: int = 8):
        self.system_prompt = system_prompt
        self.max_messages = max_messages
        self.messages = []

    def add(self, role: str, content: str):
        if role not in {"user", "assistant"}:
            raise ValueError("Role must be 'user' or 'assistant'.")
        self.messages.append({"role": role, "content": content})
        self.messages = self.messages[-self.max_messages:]

    def get_context(self):
        return [{"role": "system", "content": self.system_prompt}] + self.messages

    def clear(self):
        self.messages = []

    def message_count(self):
        return len(self.messages)


memory = ConversationMemory(
    system_prompt="You are UniMate, a helpful university study assistant.",
    max_messages=6,
)

memory.add("user", "I am studying retrieval-augmented generation.")
memory.add("assistant", "RAG retrieves relevant information and gives it to a language model.")
memory.add("user", "What is the retrieval stage?")
memory.add("assistant", "It searches the knowledge base for relevant chunks.")

print(json.dumps(memory.get_context(), indent=2))
print("Stored conversation messages:", memory.message_count())

[
  {
    "role": "system",
    "content": "You are UniMate, a helpful university study assistant."
  },
  {
    "role": "user",
    "content": "I am studying retrieval-augmented generation."
  },
  {
    "role": "assistant",
    "content": "RAG retrieves relevant information and gives it to a language model."
  },
  {
    "role": "user",
    "content": "What is the retrieval stage?"
  },
  {
    "role": "assistant",
    "content": "It searches the knowledge base for relevant chunks."
  }
]
Stored conversation messages: 4


### 4.1 Chat function with memory + RAG

The assistant retrieves relevant notes, adds them to the current question, and keeps a short conversation history. Replies are built from the retrieved notes, so everything runs offline.

In [12]:
def university_assistant_chat(user_text: str, use_rag: bool = True):
    if use_rag:
        context, sources = rag.build_context(user_text, top_k=3)
        user_content = (
            f"Relevant course notes:\n{context}\n\n"
            f"Student question: {user_text}"
        )
    else:
        context, sources = "", []
        user_content = user_text

    memory.add("user", user_content)

    if use_rag and context:
        response = "Here is what I found in the course notes:\n\n" + context
    elif use_rag:
        response = "I couldn't find relevant information in the course notes."
    else:
        response = "RAG is turned off, so I have no course notes to search."

    memory.add("assistant", response)
    return response, sources


demo_response, demo_sources = university_assistant_chat(
    "What does a typical RAG pipeline include?"
)
print(demo_response)
print("\nSources:", [x["source"] for x in demo_sources])

Here is what I found in the course notes:

Source: RAG_Week3
Retrieval-augmented generation (RAG) retrieves relevant external information and provides it to a language model as context. A typical RAG pipeline includes ingestion, chunking, retrieval, prompt construction, and generation.

Sources: ['RAG_Week3']


## 5. UniMate dashboard

A compact notebook interface for asking questions, toggling RAG, viewing retrieved sources, and clearing the conversation. It uses `ipywidgets`, so it works inside Jupyter Notebook / JupyterLab environments that support widgets.

In [13]:
import ipywidgets as widgets
from IPython.display import display, clear_output

question_box = widgets.Textarea(
    value="",
    placeholder="Ask a question about your university notes...",
    description="Question:",
    layout=widgets.Layout(width="100%", height="90px")
)
rag_toggle = widgets.Checkbox(value=True, description="Use RAG")
send_button = widgets.Button(description="Ask UniMate", button_style="primary", icon="paper-plane")
clear_button = widgets.Button(description="Clear chat", icon="trash")
chat_output = widgets.Output(layout={"border": "1px solid #ddd", "padding": "10px"})
status = widgets.HTML(value="<b>UniMate is ready.</b>")

def on_send_clicked(_):
    question = question_box.value.strip()
    if not question:
        status.value = "<span style='color:#b45309'>Please enter a question.</span>"
        return

    send_button.disabled = True
    status.value = "<i>Thinking...</i>"
    with chat_output:
        print("You:", question)
        try:
            response, sources = university_assistant_chat(
                question, use_rag=rag_toggle.value
            )
            print("UniMate:", response)
            if sources:
                print("Sources:", ", ".join(sorted({s["source"] for s in sources})))
            print("-" * 60)
        except Exception as exc:
            print("Error:", exc)
    question_box.value = ""
    status.value = "<b>Ready for your next question.</b>"
    send_button.disabled = False

def on_clear_clicked(_):
    memory.clear()
    chat_output.clear_output()
    status.value = "<b>Conversation memory cleared.</b>"

send_button.on_click(on_send_clicked)
clear_button.on_click(on_clear_clicked)

dashboard = widgets.VBox([
    widgets.HTML("<h2>UniMate — University Assistant</h2><p>Ask questions about the sample course notes.</p>"),
    question_box,
    widgets.HBox([rag_toggle, send_button, clear_button]),
    status,
    chat_output,
])
display(dashboard)

## 6. Quick self-check

Run this final cell to verify that the core components work.

In [14]:
# 1) Chunking should produce at least one chunk.
assert len(chunks) > 0

# 2) The registry should contain both tools.
assert "search_course_notes" in registry.tools
assert "calculate" in registry.tools

# 3) Calculator tool should return the expected result.
assert registry.execute("calculate", '{"expression": "6 * 7"}') == 42

# 4) Retrieval should return at least one result for a known topic.
assert len(rag.retrieve("supervised learning", top_k=2)) > 0

# 5) Memory should include a system message.
assert memory.get_context()[0]["role"] == "system"

print("All self-checks passed.")

All self-checks passed.


## Project structure explained

| Component | What it does |
|---|---|
| `SearchNotesArgs`, `CalculatorArgs` | Pydantic schemas for safe, structured tool arguments |
| `ToolRegistry` | Registers tools and validates their inputs before execution |
| Parallel execution | Runs independent tool calls concurrently |
| `ingest_documents()` | Cleans and splits source documents into chunks |
| `RAGService` | Retrieves relevant chunks and builds context for answering |
| `evaluate_retrieval()` | Measures Recall@K and MRR on a small test set |
| `ConversationMemory` | Keeps a limited recent conversation history |
| Dashboard | Provides a simple interactive interface in the notebook |

### Suggested extensions
- Replace TF-IDF with sentence-transformer embeddings (can run locally).
- Store chunks in FAISS or Chroma.
- Add PDF ingestion.
- Add a summarization-based memory for older conversation turns.
- Add more tools, such as assignment deadline lookup or course schedule search.
